# Day 18: SQL 窗口函数 —— CTE + ROW_NUMBER/RANK/LEAD/LAG

> **目标**: 先理解 CTE（临时命名子查询），再掌握 SQL 最强大的分析功能——窗口函数。
> **环境**: DuckDB (`%%sql`)
> **数据**: `../data/sales.csv` + `../data/customers.csv`

In [3]:
import duckdb 

%load_ext sql
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. CTE 基础 —— 临时命名子查询

**为什么用 CTE**:
- 把复杂查询拆成可读的小块（像 Python 的变量赋值）
- 同一个子查询可以复用多次
- 避免嵌套过深的子查询

**语法**:
```sql
WITH 名字 AS (
  SELECT ... FROM ...
)
SELECT * FROM 名字;
```

**类比 Python**:
```python
clean_data = df.copy()  # ← CTE
clean_data['product'] = df['product'].str.lower()
result = clean_data.groupby('country').sum()  # ← 主查询
```

In [4]:
%%sql
-- 基础 CTE：先清洗数据，再分组
WITH clean_sales AS (
  SELECT
    order_id,
    customer_id,
    TRIM(LOWER(product)) AS product_clean,
    total,
    country,
    order_date
  FROM '../data/sales.csv'
)
SELECT
  product_clean,
  COUNT(*) AS cnt,
  SUM(total) AS total_sales
FROM clean_sales
GROUP BY product_clean
ORDER BY total_sales DESC
LIMIT 5;

Running query in 'duckdb:///:memory:'

product_clean,cnt,total_sales
phone,93,257831
keyboard,99,225111
mouse,81,216155
headphones,67,204686
laptop,87,200939


In [5]:
%%sql
-- 多个 CTE：用逗号分隔，像 Python 的多个变量
WITH sales_clean AS (
  SELECT order_id, customer_id, total, country, order_date
  FROM '../data/sales.csv'
),
country_stats AS (
  SELECT country, COUNT(*) AS cnt, SUM(total) AS total_sales
  FROM sales_clean
  GROUP BY country
)
SELECT * FROM country_stats
WHERE total_sales > 50000
ORDER BY total_sales DESC;

Running query in 'duckdb:///:memory:'

country,cnt,total_sales
UK,197,534817
US,125,300613
France,80,208165
Germany,66,144020
China,32,80101


## 2. 窗口函数入门

**窗口函数 = 聚合函数 + 不改变行数**。

普通 `GROUP BY` 每组变成一行（降维），窗口函数每行保留但附加一个统计值（不改行数）。

**语法**:
```sql
函数() OVER (PARTITION BY 分组列 ORDER BY 排序列) AS 别名
```

- `PARTITION BY` = 按什么分组（类似 GROUP BY）
- `ORDER BY` = 组内排序（可选）
- `OVER()` = 窗口框架

## 3. ROW_NUMBER —— 组内排名（不重复）

`ROW_NUMBER()` 给每组内的行按顺序编号，1, 2, 3... 不会重复。

In [6]:
%%sql
-- 每个国家内，按订单额降序排名
SELECT
  order_id,
  country,
  total,
  ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn
FROM '../data/sales.csv'
LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,country,total,rn
O1238,US,9995,1
O1470,US,9995,2
O1432,US,9995,3
O1117,US,7996,4
O1092,US,7996,5
O1083,US,6495,6
O1218,US,6495,7
O1160,US,6495,8
O1234,US,6495,9
O1390,US,5997,10


In [7]:
%%sql
-- 每个国家取 TOP 3 订单（用 CTE + WHERE 筛选）
WITH ranked AS (
  SELECT
    order_id, country, total,
    ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn
  FROM '../data/sales.csv'
)
SELECT * FROM ranked
WHERE rn <= 3
ORDER BY country, rn;

Running query in 'duckdb:///:memory:'

order_id,country,total,rn
O1237,China,6495,1
O1035,China,6495,2
O1127,China,5997,3
O1483,France,9995,1
O1319,France,9995,2
O1463,France,7996,3
O1370,Germany,9995,1
O1342,Germany,9995,2
O1131,Germany,7996,3
O1152,UK,9995,1


## 4. RANK / DENSE_RANK —— 处理并列

| 函数 | 并列处理 | 序号跳跃 | 示例 (100, 100, 90) |
|------|----------|----------|---------------------|
| `ROW_NUMBER` | 随机分配 | 无 | 1, 2, 3 |
| `RANK` | 同排名 | 有 (1,1,3) | 1, 1, 3 |
| `DENSE_RANK` | 同排名 | 无 (1,1,2) | 1, 1, 2 |

In [8]:
%%sql
-- 对比三种排名函数
SELECT
  order_id,
  country,
  total,
  ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn,
  RANK() OVER (PARTITION BY country ORDER BY total DESC) AS rnk,
  DENSE_RANK() OVER (PARTITION BY country ORDER BY total DESC) AS drnk
FROM '../data/sales.csv'
WHERE country = 'US'
ORDER BY total DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,country,total,rn,rnk,drnk
O1238,US,9995,1,1,1
O1470,US,9995,2,1,1
O1432,US,9995,3,1,1
O1117,US,7996,4,4,2
O1092,US,7996,5,4,2
O1083,US,6495,6,6,3
O1218,US,6495,7,6,3
O1160,US,6495,8,6,3
O1234,US,6495,9,6,3
O1390,US,5997,10,10,4


## 5. LEAD / LAG —— 前后行取值

- `LEAD(col, n)` = 取后第 n 行的值
- `LAG(col, n)` = 取前第 n 行的值

常用于：计算环比、相邻行差异。

In [9]:
%%sql
-- 按月份统计，计算销售额的环比变化
WITH monthly AS (
  SELECT
    STRFTIME('%Y-%m', order_date) AS ym,
    SUM(total) AS total_sales
  FROM '../data/sales.csv'
  GROUP BY ym
  ORDER BY ym
)
SELECT
  ym,
  total_sales,
  LAG(total_sales, 1) OVER (ORDER BY ym) AS prev_month,
  total_sales - LAG(total_sales, 1) OVER (ORDER BY ym) AS change,
  ROUND(
    (total_sales - LAG(total_sales, 1) OVER (ORDER BY ym)) * 100.0 / LAG(total_sales, 1) OVER (ORDER BY ym),
    1
  ) AS change_pct
FROM monthly;

Running query in 'duckdb:///:memory:'

ym,total_sales,prev_month,change,change_pct
2024-01,99365,None,None,None
2024-02,81795,99365,-17570,-17.7
2024-03,133371,81795,51576,63.1
2024-04,123882,133371,-9489,-7.1
2024-05,71176,123882,-52706,-42.5
2024-06,119061,71176,47885,67.3
2024-07,101271,119061,-17790,-14.9
2024-08,104388,101271,3117,3.1
2024-09,103183,104388,-1205,-1.2
2024-10,117265,103183,14082,13.6


## 6. 聚合函数 + OVER —— 不改行数的聚合

`SUM() OVER`, `AVG() OVER`, `COUNT() OVER`... 在窗口上做聚合，保留所有行。

In [10]:
%%sql
-- 每行显示该国家的总销售额（不改行数）
SELECT
  order_id,
  country,
  total,
  SUM(total) OVER (PARTITION BY country) AS country_total,
  ROUND(total * 100.0 / SUM(total) OVER (PARTITION BY country), 1) AS pct
FROM '../data/sales.csv'
LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,country,total,country_total,pct
O1005,UK,198,534817,0.0
O1006,UK,6495,534817,1.2
O1007,UK,1198,534817,0.2
O1008,UK,1495,534817,0.3
O1010,UK,1797,534817,0.3
O1012,UK,897,534817,0.2
O1013,UK,1299,534817,0.2
O1019,UK,897,534817,0.2
O1023,UK,4995,534817,0.9
O1025,UK,1797,534817,0.3


## 今日要点总结

| 概念 | 语法 | 作用 | 行数变化 |
|------|------|------|----------|
| CTE | `WITH name AS (SELECT ...)` | 临时命名子查询 | 不变 |
| ROW_NUMBER | `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` | 组内编号（不重复） | 不变 |
| RANK | `RANK() OVER (...)` | 组内排名（并列跳号） | 不变 |
| DENSE_RANK | `DENSE_RANK() OVER (...)` | 组内排名（并列不跳号） | 不变 |
| LEAD | `LEAD(col, n) OVER (ORDER BY ...)` | 取后第 n 行 | 不变 |
| LAG | `LAG(col, n) OVER (ORDER BY ...)` | 取前第 n 行 | 不变 |
| SUM OVER | `SUM(col) OVER (PARTITION BY ...)` | 组内求和（每行显示） | 不变 |

**核心心法**: CTE = SQL 的「中间变量」，窗口函数 = 不改行数的聚合。两者结合，可以写出极其优雅的分析查询。